In [ ]:
# ============================================================================
# ENTERPRISE HR CHATBOT - FAST & CLEAN EDITION
# ============================================================================

import os
import sys
import logging
import warnings
import textwrap
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# 1. SUPPRESS UGLY WARNINGS
# ----------------------------------------------------------------------------
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

# LangChain & AI Libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from openai import OpenAI as OpenAIClient

# Load environment variables
load_dotenv()

True

In [2]:
# ============================================================================
# 2. CONFIGURATION (SPEED OPTIMIZED)
# ============================================================================
class SystemConfig(BaseModel):
    file_path: str = Field(default="Egyptian_ERP_System_Comprehensive_Manual.pdf", description="Path to PDF")
    
    # AI SETTINGS
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    
    # SPEED HACK: Smaller chunks = Faster LLM processing
    # We keep k=10 to find the page, but read less text per page.
    chunk_size: int = 600 
    chunk_overlap: int = 100
    
    # LM STUDIO SETTINGS
    lm_studio_base_url: str = "http://localhost:1234/v1"
    lm_studio_api_key: str = "lm-studio"
    lm_studio_model: str = "llama-3.2-3b-instruct"
    temperature: float = 0.3
    
    # RETRIEVAL SETTINGS
    similarity_k: int = 10 

    class Config:
        validate_assignment = True

In [ ]:
# ============================================================================
# 3. LOGGING SETUP
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[
        logging.FileHandler('hr_chatbot.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [ ]:
# ============================================================================
# 4. LLM CLIENT (LM STUDIO ADAPTER)
# ============================================================================
class LocalLLM:
    def __init__(self, base_url: str, api_key: str, model: str, temperature: float = 0.7):
        self.client = OpenAIClient(base_url=base_url, api_key=api_key)
        self.model = model
        self.temperature = temperature

    def predict(self, prompt: str) -> str:
        messages = [
            {"role": "system", "content": "You are a professional HR Assistant. Format your answers clearly using Markdown. Focus on HR policies, leave management, employee benefits, and company procedures."},
            {"role": "user", "content": prompt}
        ]
        try:
            completion = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=self.temperature,
                timeout=90.0
            )
            return completion.choices[0].message.content or ""
        except Exception as e:
            return f"Error connecting to LM Studio: {e}"

In [ ]:
# ============================================================================
# 5. MAIN CHATBOT CLASS
# ============================================================================
class EnterpriseHRChatbot:
    def __init__(self, config: SystemConfig):
        self.config = config
        self.vector_db = None
        self.embeddings = None
        self.llm = None
        
        logger.info("--- INITIALIZING HR CHATBOT (Final Edition) ---")
        self._ingest_documents()
        self._init_model()
        logger.info("--- READY ---")

    def _ingest_documents(self):
        try:
            if not os.path.exists(self.config.file_path):
                raise FileNotFoundError(f"PDF not found: {self.config.file_path}")

            logger.info(f"Loading PDF: {self.config.file_path}...")
            loader = PyPDFLoader(self.config.file_path)
            docs = loader.load()
            
            # Filter tiny pages
            docs = [d for d in docs if len(d.page_content) > 50]
            
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.config.chunk_size, 
                chunk_overlap=self.config.chunk_overlap
            )
            chunks = splitter.split_documents(docs)
            
            logger.info("Building Vector Index...")
            self.embeddings = HuggingFaceEmbeddings(model_name=self.config.embedding_model)
            self.vector_db = FAISS.from_documents(chunks, self.embeddings)
            
        except Exception as e:
            logger.error(f"Ingestion failed: {e}")
            sys.exit(1)

    def _init_model(self):
        self.llm = LocalLLM(
            base_url=self.config.lm_studio_base_url,
            api_key=self.config.lm_studio_api_key,
            model=self.config.lm_studio_model,
            temperature=self.config.temperature
        )

    def query(self, question: str):
        # 1. Retrieve (Get top 10 chunks)
        docs = self.vector_db.similarity_search(question, k=self.config.similarity_k)
        
        # Build Context String
        context = "\n".join([d.page_content for d in docs])
        
        # 2. Prompt with HR Formatting Instructions
        prompt = f"""You are a helpful HR Assistant for an Egyptian ERP system. Answer the question based strictly on the context below. Focus on HR policies, leave management, employee benefits, attendance rules, and company procedures.

FORMATTING RULES:
1. Start with a direct answer.
2. Use **Bold Headers** for key sections.
3. Use bullet points (•) for lists.
4. Keep paragraphs short.
5. Provide accurate, policy-based guidance.

CONTEXT:
{context}

QUESTION: 
{question}

ANSWER:"""
        
        # 3. Generate
        answer = self.llm.predict(prompt)
        
        return answer

In [6]:

# ============================================================================
# 6. BEAUTIFUL OUTPUT PRINTER (SOURCES REMOVED)
# ============================================================================
def print_boxed(title, text):
    width = 135 # Adjusted width for readability
    
    # Border characters
    horizontal = "═" * (width - 2)
    top_border = "╔" + horizontal + "╗"
    bottom_border = "╚" + horizontal + "╝"
    divider = "╠" + horizontal + "╣"
    
    print("\n" + top_border)
    print(f"║ {title.center(width - 4)} ║")
    print(divider)
    
    wrapper = textwrap.TextWrapper(
        width=width-6, 
        initial_indent="  ", 
        subsequent_indent="    ",
        break_long_words=False,
        replace_whitespace=False
    )
    
    for line in text.split('\n'):
        clean_line = line.strip()
        if clean_line.startswith('**'):
             print(f"║  {clean_line:<{width-6}}  ║")
        elif clean_line.startswith('-') or clean_line.startswith('•') or clean_line.startswith('*'):
             wrapped_bullets = wrapper.wrap(clean_line)
             for i, w_line in enumerate(wrapped_bullets):
                 if i == 0:
                     print(f"║ {w_line:<{width-4}} ║")
                 else:
                     print(f"║     {w_line.strip():<{width-8}} ║")
        elif clean_line == "":
             print(f"║ {' ':<{width-4}} ║")
        else:
            for wrapped_line in wrapper.wrap(line):
                print(f"║ {wrapped_line:<{width-4}} ║")
    
    print(bottom_border + "\n")


In [7]:
pip install --upgrade openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install arabic-reshaper python-bidi

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================================
# 7. RUNNER
# ============================================================================
if __name__ == "__main__":
    bot = EnterpriseHRChatbot(SystemConfig())
    
    questions = [
        "What is the policy on sick leave?",
        "How many days of casual leave am I entitled to?",
        "What is the maternity leave policy?",
        "What are the attendance rules and working hours?",
        "How does the GOSI (Social Insurance) contribution work for employees?"
    ]
    
    print("\n" + "="*80)
    print(f"STARTING BATCH TEST ({len(questions)} Questions)")
    print("="*80)

    for i, q in enumerate(questions, 1):
        print(f"\n QUESTION {i}: {q}")
        
        # Query the bot
        answer = bot.query(q)
        
        # Print Beautiful Result (No Sources)
        print_boxed(f"ANSWER {i}", answer)

2026-02-22 04:00:38,646 - --- INITIALIZING Customer Support CHATBOT (Final Edition) ---
2026-02-22 04:00:38,648 - Loading PDF: Egyptian_ERP_System_Comprehensive_Manual.pdf...
2026-02-22 04:00:59,428 - Building Vector Index...
2026-02-22 04:03:05,718 - Loading faiss with AVX2 support.
2026-02-22 04:03:06,101 - Successfully loaded faiss with AVX2 support.
2026-02-22 04:03:07,383 - --- READY ---



STARTING BATCH TEST (5 Questions)

 QUESTION 1: I forgot my password. How do I reset it to access the system?


2026-02-22 04:03:39,045 - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                                               ANSWER 1                                                              ║
╠═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  **Forgot Password Procedure**                                                                                                      ║
║                                                                                                                                     ║
║   If you forget your password, click the Forgot Password link on the login page. Enter your username or email address, and a        ║
║     password reset link will be sent to your registered email. Follow the link to create a new password.                            ║
║                                              

2026-02-22 04:03:53,953 - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


...

2026-02-22 04:04:28,639 - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                                               ANSWER 4                                                              ║
╠═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  **Understanding Three-Way Matching**                                                                                               ║
║                                                                                                                                     ║
║   Three-Way Matching is a control process in the Accounts Payable (AP) module that verifies the accuracy of vendor invoices by      ║
║     comparing them with purchase orders (POs) and goods receipts (GRNs).                                                            ║
║                                              

2026-02-22 04:04:36,278 - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                                               ANSWER 5                                                              ║
╠═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  **Inventory Valuation Methods**                                                                                                    ║
║                                                                                                                                     ║
║   The Egyptian ERP System supports the following inventory valuation methods:                                                       ║
║                                                                                                                                     ║
║   • **FIFO (First In, First Out)**: oldest in

: 